In [2]:
import os
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from functions_javaScript import getMyParcels_fxn,getParcelById_fxn ,getParcelByName_fxn,getParcelByStatus_fxn,generateInvoice_fxn,delayedParcels_fxn,getLatestParcel_fxn,getParcelByCategory_fxn

In [3]:
# initilize the model
model=ChatGroq(
    model="qwen/qwen3-32b"
)

In [4]:
# define all the tools 
@tool 
def getParcelByStatus(status:str)->str:
    """
    Get parcels by shipment status.
    Allowed values (must be exact, uppercase):
    PLACED, IN_TRANSIT, DISPATCHED, OUT_FOR_DELIVERY, DELIVERED, CANCELLED
    Use when the user asks for parcels by status.
    Convert natural language to one of the allowed values before calling.
    Do NOT use for delayed/late queries (use delayedParcels instead).
    """ 
    # return "getparcelByStatus"
    return getParcelByStatus_fxn(status)   #IN_TRANSIT

@tool
def getParcelByName(name:str)->str:
    """
    Get parcels by product name.
    Use when the user mentions a product instead of a parcel ID.
    Pass the exact name as given by the user:
    - Do NOT modify, shorten, or normalize words
    - Keep full phrases and spaces intact
    If the input matches a valid parcel ID format, use getParcelById instead.
    """
    # return "GetParcelByName"
    return getParcelByName_fxn(name)   #home furniture

@tool
def getMyParcels()->str:
    """
    Get all parcels of the current user.
    Use when the user asks for all their parcels or shipments.
    Do NOT use if the user specifies filters like status, category, or name.
    """
    return getMyParcels_fxn()

@tool
def getParcelById(parcelId:str)->str:
    """
    Get parcel details using a parcel ID.
    A valid parcelId must follow this format:
    starts with "P-IDX" and is followed by exactly 7 characters.
    Use this tool ONLY if the input strictly matches this format.
    If the input does not match this pattern, use getParcelByName instead.
    """
    return getParcelById_fxn(parcelId)   #69d398da7cdc334aebd5ed7f

@tool
def generateInvoice(parcelId:str)->str:
    """
    Generate an invoice for a parcel.
    Use when the user asks for invoice, bill, or receipt.
    Input can be:
    - A valid parcelId (starts with "P-IDX" + 7 characters)
    - OR an exact product name (do not modify input)
    Do NOT guess or alter the input..
    """
    # return "GenerateInvoice"
    return generateInvoice_fxn(parcelId)  #69d398da7cdc334aebd5ed7f

@tool
def delayedParcels()->str:
    """
    Get all delayed parcels.
    Use when the user asks about delayed, late, overdue,
    or not delivered on time shipments.
    Do NOT use getParcelByStatus for delay-related queries.
    """
    # return "delaysedParcels"
    return delayedParcels_fxn()

@tool
def getLatestParcel()->str:
    """ 
    Get the most recently created parcel.
    Use when the user asks for latest, last, or most recent parcel.
    Do NOT use if the user is asking for multiple or filtered parcels
    """
    # return "getLatestparcel"
    return getLatestParcel_fxn()

@tool
def getParcelByCategory(category:str)->str:
    """
    Get parcels by category.
    Allowed values (must be exact, lowercase):
    electronics, clothing, documents, food, furniture, medical,
    automotive, cosmetics, sports, books, fragile, industrial
    Use when the user asks for parcels by category.
    Convert user input into one of the allowed values before calling.
    Do NOT call if the category cannot be clearly mapped.
    """
    # return "getParcelByCategory"
    return getParcelByCategory_fxn(category)   #clothing


In [5]:
#init the system prompt
sys_prompt=SystemMessage(content="""
You are a helpful logistics assistant with access to tools.
Your job:
- Understand the user query
- Decide whether a tool is needed
- If needed, select and call the most appropriate tool
- Use the tool result to generate a clear, human-friendly response
Guidelines:
- Use tools ONLY when the query requires parcel or invoice data
- For greetings, casual conversation, or general questions, respond directly without using tools
- Choose the most specific tool available when needed
- Do not guess or invent any data
- Follow all tool input rules strictly
Response rules:
- Convert tool results into natural, user-friendly text
- Do NOT show JSON, tool calls, or internal reasoning
- Keep responses clear and concise
If no tool is appropriate:
- Respond conversationally or ask a clear follow-up question
- If required inputs are missing, ask the user instead of guessing
""")

In [6]:
# initize the agent
agent=create_agent(
    model=model,
    tools=[getParcelByCategory,getLatestParcel,delayedParcels,generateInvoice,getParcelById,getParcelByName,getParcelByStatus,getMyParcels],
    checkpointer=InMemorySaver(),
    system_prompt=sys_prompt
)


In [7]:
def print_llmresponce(res):
    for msg in res["messages"]:
        role = msg.__class__.__name__.replace("Message", "")
        print(f"\n[{role}]")

        if hasattr(msg, "content") and msg.content:
            print(msg.content)

        if hasattr(msg, "additional_kwargs") and msg.additional_kwargs:
            print("KWARGS:", msg.additional_kwargs)

def jarvis(msg:str,id:int=1):
    llm_res=agent.invoke({"messages":[HumanMessage(content=msg)]},{"configurable":{"thread_id":str(id)}})
    print_llmresponce(llm_res)


In [9]:
jarvis("my name is gaurav ")


[Human]
my name is gaurav 

[AI]
Hello, Gaurav! How can I assist you today?
KWARGS: {'reasoning_content': 'Okay, the user said, "my name is gaurav." Let me think about how to respond.\n\nFirst, I need to check if any of the provided tools are relevant here. The user is providing their name, which is a personal detail. Looking at the tools, they are all related to parcels, invoices, and shipment statuses. None of the tools require the user\'s name as an input parameter. The functions like getMyParcels might be used later if Gaurav asks about his parcels, but just providing the name doesn\'t trigger any tool calls. \n\nThe guidelines say that if the query doesn\'t require parcel or invoice data, respond directly without using tools. Since the user is just introducing themselves, there\'s no need to call any functions here. My response should be a simple acknowledgment, maybe a greeting. \n\nI should make sure not to use any tools unnecessarily. The user\'s name might be useful in future

In [10]:
jarvis("please give me information of delayed parcels")


[Human]
my name is gaurav 

[AI]
Hello, Gaurav! How can I assist you today?
KWARGS: {'reasoning_content': 'Okay, the user said, "my name is gaurav." Let me think about how to respond.\n\nFirst, I need to check if any of the provided tools are relevant here. The user is providing their name, which is a personal detail. Looking at the tools, they are all related to parcels, invoices, and shipment statuses. None of the tools require the user\'s name as an input parameter. The functions like getMyParcels might be used later if Gaurav asks about his parcels, but just providing the name doesn\'t trigger any tool calls. \n\nThe guidelines say that if the query doesn\'t require parcel or invoice data, respond directly without using tools. Since the user is just introducing themselves, there\'s no need to call any functions here. My response should be a simple acknowledgment, maybe a greeting. \n\nI should make sure not to use any tools unnecessarily. The user\'s name might be useful in future

In [11]:
jarvis("yes")


[Human]
my name is gaurav 

[AI]
Hello, Gaurav! How can I assist you today?
KWARGS: {'reasoning_content': 'Okay, the user said, "my name is gaurav." Let me think about how to respond.\n\nFirst, I need to check if any of the provided tools are relevant here. The user is providing their name, which is a personal detail. Looking at the tools, they are all related to parcels, invoices, and shipment statuses. None of the tools require the user\'s name as an input parameter. The functions like getMyParcels might be used later if Gaurav asks about his parcels, but just providing the name doesn\'t trigger any tool calls. \n\nThe guidelines say that if the query doesn\'t require parcel or invoice data, respond directly without using tools. Since the user is just introducing themselves, there\'s no need to call any functions here. My response should be a simple acknowledgment, maybe a greeting. \n\nI should make sure not to use any tools unnecessarily. The user\'s name might be useful in future

In [12]:
jarvis("what was my name ")


[Human]
my name is gaurav 

[AI]
Hello, Gaurav! How can I assist you today?
KWARGS: {'reasoning_content': 'Okay, the user said, "my name is gaurav." Let me think about how to respond.\n\nFirst, I need to check if any of the provided tools are relevant here. The user is providing their name, which is a personal detail. Looking at the tools, they are all related to parcels, invoices, and shipment statuses. None of the tools require the user\'s name as an input parameter. The functions like getMyParcels might be used later if Gaurav asks about his parcels, but just providing the name doesn\'t trigger any tool calls. \n\nThe guidelines say that if the query doesn\'t require parcel or invoice data, respond directly without using tools. Since the user is just introducing themselves, there\'s no need to call any functions here. My response should be a simple acknowledgment, maybe a greeting. \n\nI should make sure not to use any tools unnecessarily. The user\'s name might be useful in future

In [13]:
jarvis("give me details of 5 latest parcels")


[Human]
my name is gaurav 

[AI]
Hello, Gaurav! How can I assist you today?
KWARGS: {'reasoning_content': 'Okay, the user said, "my name is gaurav." Let me think about how to respond.\n\nFirst, I need to check if any of the provided tools are relevant here. The user is providing their name, which is a personal detail. Looking at the tools, they are all related to parcels, invoices, and shipment statuses. None of the tools require the user\'s name as an input parameter. The functions like getMyParcels might be used later if Gaurav asks about his parcels, but just providing the name doesn\'t trigger any tool calls. \n\nThe guidelines say that if the query doesn\'t require parcel or invoice data, respond directly without using tools. Since the user is just introducing themselves, there\'s no need to call any functions here. My response should be a simple acknowledgment, maybe a greeting. \n\nI should make sure not to use any tools unnecessarily. The user\'s name might be useful in future